# 06 — HDR by exposure normalisation

Paper Section 3.3.1: $I_i' = I_i / t_i$, $I_{\rm HDR} = \frac1N \sum_i I_i'$ over the nine
stacked exposures, per polariser.  `config.CHANNEL_SHIFTS` (non‑zero only in
legacy mode) is applied to the stacked planes first.

Output: `products/hdr/hdr_expnorm_pol{1,2,3}.fits` + a preview PNG.
Legacy source: `make_exposure_norm_HDR_from_stacked_exposures_Chaitanya.ipynb`.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
from scipy import ndimage
template = fits.getheader(utils.stacked_filename(config.INV_EXPOSURES[0]))
hdr_planes = {}
for i, pos in enumerate(config.POLARIZER_POSITIONS):
    t0 = time.time()
    imgs = []
    for inv_exp in config.INV_EXPOSURES:
        d = np.asarray(fits.getdata(utils.stacked_filename(inv_exp), memmap=True)[i], dtype=np.float32)
        imgs.append(ndimage.shift(d, config.CHANNEL_SHIFTS[pos], order=1) if any(config.CHANNEL_SHIFTS[pos]) else d)
    hdr_planes[pos] = utils.exposure_normalization_stacking(imgs, config.EXPOSURE_TIMES_S)
    del imgs
    fits.writeto(utils.hdr_filename("expnorm", pos), hdr_planes[pos], header=utils.hdr_header(template, pos, "expnorm"), overwrite=True)
    print(f"pol{pos}: min {hdr_planes[pos].min():.4g} max {hdr_planes[pos].max():.4g}  ({time.time()-t0:.0f} s)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, pos in zip(axes, config.POLARIZER_POSITIONS):
    ax.imshow(utils.asinh_stretch(hdr_planes[pos][::4, ::4]), cmap="gray"); ax.set_title(f"expnorm HDR pol{pos} (asinh)"); ax.axis("off")
plt.tight_layout()

In [ ]:
from PIL import Image
rgb = np.stack([utils.normalise_channel(hdr_planes[p]) for p in config.POLARIZER_POSITIONS], axis=-1)
Image.fromarray((rgb * 255).astype(np.uint8)).save(config.HDR_DIR / "hdr_expnorm_preview.png")
plt.figure(figsize=(8, 5.5)); plt.imshow(rgb[::4, ::4]); plt.axis("off"); plt.title("expnorm HDR preview (pol1 -> R, pol2 -> G, pol3 -> B)")